# 02 · Descriptive statistics (Section 3.3 and Appendix D)

Fills the placeholders of Section 3.3 and the descriptive appendix items. Requires `01_constructs_and_checks.ipynb` to have been run.

| Thesis item | Output |
|---|---|
| Table 2 – sample composition by subprocess | `outputs/1_ch3_descriptives/table02_sample_composition.csv` |
| Table 3 – task-level variables, Release 5 | `outputs/1_ch3_descriptives/table03_task_variables_R5.csv` |
| Section 3.3 paragraph (i)–(iii) | `outputs/1_ch3_descriptives/section3_3_facts.txt` |
| Table 3, Release 4 version (Appendix D) | `outputs/4_appendix_D/tableD9_task_variables_R4.csv` |
| Table D3 – correlations of the Table 7–8 regressors | `outputs/4_appendix_D/tableD3_regressor_correlations.csv` |
| Table D4 – top-10 task share of usage | `outputs/4_appendix_D/tableD4_top10_task_share.csv` |
| Figure D3 – human-only time by platform, R5 | `outputs/4_appendix_D/figD3_human_only_time_by_platform_R5.png` |

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import r2r_common as rc

pd.set_option("display.width", 160)
tm, panel = rc.load_data()
task_exp = rc.read_construct("task_exposure.csv")
task_success = rc.read_construct("task_success.csv")
idx = rc.read_construct("r2r_indices.csv")
t = tm.set_index("task_id")

[load] task_master: 212 rows, 91 in scope ({'Master Data Management': 8, 'Record': 31, 'Close & Consolidation': 5, 'Report': 13, 'Analysis & Business Insight': 12, 'Planning & Budgeting': 22})
[load] panel: 9,522 rows, releases [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)], 50 tasks; rows whose task is not in scope: 0


## Table 2 – Sample composition by subprocess

Share of process time sums raw task time fractions; mean expertise and wage are time-fraction-weighted; covered = task appears in the usage data of the platform in that release.

In [2]:
rows = []
for scope in rc.SUBPROCESS_ORDER + ["Entire Process"]:
    ids = rc.scope_task_ids(tm, scope)
    row = {"Subprocess": scope, "In-scope tasks (n)": len(ids),
           "Share of process time (%)": 100 * t.loc[ids, "task_time_fraction"].sum() / t["task_time_fraction"].sum(),
           "Mean expertise (yrs)": rc._weighted_mean(t.loc[ids, "ridge_task_years"], t.loc[ids, "task_time_fraction"]),
           "Mean wage (USD/h)": rc._weighted_mean(t.loc[ids, "wage_hourly_mean"], t.loc[ids, "task_time_fraction"]),
           "Tasks with beta >= 0.5 (n)": int((t.loc[ids, "beta"] >= rc.BETA_CAPABLE).sum())}
    for r in (4, 5):
        cov = {p: len(set(ids) & rc.get_covered_task_ids(panel, r, p)) for p in rc.PLATFORMS}
        row[f"Tasks covered R{r} (C.ai / API)"] = f"{cov['claude_ai']} / {cov['1p_api']}"
    rows.append(row)
table2 = pd.DataFrame(rows).round(2)
rc.save_csv(table2, "ch3", "table02_sample_composition.csv")
table2

[write] outputs/1_ch3_descriptives/table02_sample_composition.csv

,Subprocess,In-scope tasks (n),Share of process time (%),Mean expertise (yrs),Mean wage (USD/h),Tasks with beta >= 0.5 (n),Tasks covered R4 (C.ai / API),Tasks covered R5 (C.ai / API)
0,Master Data Management,8,5.36,14.89,66.25,8,5 / 5,6 / 5
1,Record,31,33.98,13.05,29.17,30,13 / 8,12 / 11
2,Close & Consolidation,5,4.16,13.65,33.88,5,2 / 2,2 / 2
3,Report,13,17.49,14.40,49.47,13,2 / 2,4 / 2
4,Analysis & Business Insight,12,15.00,15.19,65.52,12,5 / 5,5 / 6
5,Planning & Budgeting,22,23.99,15.35,59.40,22,8 / 4,9 / 2
6,Entire Process,91,100.00,14.28,47.61,90,35 / 26,38 / 28


## Table 3 – Descriptive statistics of task-level variables (Release 5; Release 4 version for Appendix D)

Statistics are over tasks with a non-missing value on the indicated platform. A skewness column is added to document (iii) in the Section 3.3 paragraph (it can be dropped from the printed table).

In [3]:
def describe(name, platform, s):
    s = pd.Series(s, dtype=float).dropna()
    return {"Variable": name, "Platform": platform, "N": len(s), "Mean": s.mean(), "SD": s.std(),
            "Min": s.min(), "Median": s.median(), "Max": s.max(), "Skewness": s.skew()}


def table3(release):
    ex = task_exp[(task_exp["release"] == release) & (task_exp["gate_on"])].set_index("task_id")
    used = ex[ex["WorkUsage"] > 0]
    ts = task_success[task_success["release"] == release].set_index("task_id")
    ix = idx[idx["release"] == release].set_index("task_id")
    out = [describe("Task time fraction (hours/week, raw)", "—", t["task_time_fraction"]),
           describe("Ridge task expertise (years)", "—", t["ridge_task_years"]),
           describe("Task value (USD/hour)", "—", t["wage_hourly_mean"]),
           describe("Theoretical exposure (beta)", "—", t["beta"]),
           describe("Work usage (conversations)", "Pooled", used["WorkUsage"]),
           describe("Automation factor (alpha)", "Pooled", used["alpha"]),
           describe("Task observed exposure (r~), gate = ON", "Pooled", ex["r_tilde"])]
    for p in rc.PLATFORMS:
        out.append(describe("Task success (%)", rc.PLATFORM_LABELS[p], ts[p] if p in ts else []))
    for p in rc.PLATFORMS:
        out.append(describe("Human-only time (hours)", rc.PLATFORM_LABELS[p],
                            rc.get_task_metric(panel, release, p, "human_only_time")))
    for p in rc.PLATFORMS:
        out.append(describe("AI autonomy (1 to 5)", rc.PLATFORM_LABELS[p], rc.get_task_metric(panel, release, p, "ai_autonomy")))
    with np.errstate(divide="ignore"):
        for col, label in [("input_index", "ln(input context-length index)"),
                           ("output_index", "ln(output context-length index)"), ("cost_index", "ln(cost index)")]:
            out.append(describe(label, "1P API", np.log(ix[col].where(ix[col] > 0))))
    return pd.DataFrame(out).round(3)


table3_r5 = table3(5)
rc.save_csv(table3_r5, "ch3", "table03_task_variables_R5.csv")
rc.save_csv(table3(4), "appD", "tableD9_task_variables_R4.csv")
table3_r5

[write] outputs/1_ch3_descriptives/table03_task_variables_R5.csv
[write] outputs/4_appendix_D/tableD9_task_variables_R4.csv


,Variable,Platform,N,Mean,SD,Min,Median,Max,Skewness
0,"Task time fraction (hours/week, raw)",—,91,3.503,2.127,0.583,3.233,11.000,1.414
1,Ridge task expertise (years),—,91,14.235,1.455,9.298,14.466,17.109,-0.595
2,Task value (USD/hour),—,91,49.441,29.480,24.550,45.560,129.630,1.362
3,Theoretical exposure (beta),—,91,0.659,0.246,0.000,0.500,1.000,0.502
4,Work usage (conversations),Pooled,38,275.895,616.842,15.000,94.000,3616.000,4.626
5,Automation factor (alpha),Pooled,38,0.766,0.163,0.500,0.810,1.000,-0.566
6,"Task observed exposure (r~), gate = ON",Pooled,91,0.167,0.341,0.000,0.000,1.000,1.584
7,Task success (%),Claude.ai,31,68.417,12.065,50.000,69.343,88.889,0.068
8,Task success (%),1P API,20,54.292,12.909,24.000,53.860,79.310,-0.223
9,Human-only time (hours),Claude.ai,38,3.571,1.612,0.822,3.331,7.676,0.563


## Section 3.3 paragraph – (i) Record's task count vs time share, (ii) success readings by platform, (iii) skewness of usage

In [4]:
lines = []
rec = table2.set_index("Subprocess").loc["Record"]
lines.append(f"(i) Record holds {rec['In-scope tasks (n)']} of {len(tm)} in-scope tasks "
             f"({100 * rec['In-scope tasks (n)'] / len(tm):.1f}% of the task count) but "
             f"{rec['Share of process time (%)']:.1f}% of process time.")
for sp in rc.SUBPROCESS_ORDER:
    r_ = table2.set_index("Subprocess").loc[sp]
    lines.append(f"    {sp}: {100 * r_['In-scope tasks (n)'] / len(tm):.1f}% of tasks vs {r_['Share of process time (%)']:.1f}% of time")
for r in (4, 5):
    ts = task_success[task_success["release"] == r]
    n_c, n_a = ts["claude_ai"].notna().sum(), ts["1p_api"].notna().sum() if "1p_api" in ts else 0
    lines.append(f"(ii) Release {r}: {n_c} tasks carry a success reading on Claude.ai and {n_a} on the 1P API "
                 f"({n_c - n_a} fewer on the API).")
for r in (4, 5):
    for p in rc.PLATFORMS:
        share = rc.get_r2r_task_pct(panel, r, p) * 100
        cnt = rc.task_series(panel, r, p, "usage", "count", agg="sum")
        lines.append(f"(iii) Release {r} {rc.PLATFORM_LABELS[p]}: usage share skewness {share.skew():.2f}, "
                     f"mean {share.mean():.2f}% vs median {share.median():.2f}%; conversation-count skewness "
                     f"{cnt.skew():.2f}, max/median {cnt.max() / cnt.median():.1f}x")
facts = "\n".join(lines)
rc.out_path("ch3", "section3_3_facts.txt").write_text(facts, encoding="utf-8")
print(facts)

(i) Record holds 31 of 91 in-scope tasks (34.1% of the task count) but 34.0% of process time.
    Master Data Management: 8.8% of tasks vs 5.4% of time
    Record: 34.1% of tasks vs 34.0% of time
    Close & Consolidation: 5.5% of tasks vs 4.2% of time
    Report: 14.3% of tasks vs 17.5% of time
    Analysis & Business Insight: 13.2% of tasks vs 15.0% of time
    Planning & Budgeting: 24.2% of tasks vs 24.0% of time
(ii) Release 4: 27 tasks carry a success reading on Claude.ai and 19 on the 1P API (8 fewer on the API).
(ii) Release 5: 31 tasks carry a success reading on Claude.ai and 20 on the 1P API (11 fewer on the API).
(iii) Release 4 Claude.ai: usage share skewness 4.60, mean 2.86% vs median 0.76%; conversation-count skewness 4.60, max/median 48.2x
(iii) Release 4 1P API: usage share skewness 2.17, mean 3.85% vs median 1.09%; conversation-count skewness 2.17, max/median 20.3x
(iii) Release 5 Claude.ai: usage share skewness 5.10, mean 2.63% vs median 0.61%; conversation-count skewn

## Table D3 – Pearson correlations among the regressors of Tables 7–8 (task × release, Releases 4–5 pooled)

Same samples as the regressions (N = 58 on Claude.ai, N = 39 on the 1P API in the thesis).

In [5]:
claude = rc.build_claude_success_dataset(tm, panel)
api = rc.build_api_success_dataset(tm, panel)
corr_c = claude[["expertise", "autonomy", "human_only_time", "task_success_pct"]].corr().round(3)
corr_a = api[["expertise", "autonomy", "ln_input_index", "ln_output_index", "task_success_pct"]].corr().round(3)
corr_c.insert(0, "platform", "Claude.ai")
corr_a.insert(0, "platform", "1P API")
tableD3 = pd.concat([corr_c, corr_a]).rename_axis("variable").reset_index()
rc.save_csv(tableD3, "appD", "tableD3_regressor_correlations.csv")
print("N Claude.ai =", len(claude), "| N 1P API =", len(api))
tableD3

[write] outputs/4_appendix_D/tableD3_regressor_correlations.csv
N Claude.ai = 58 | N 1P API = 39


,variable,platform,expertise,autonomy,human_only_time,task_success_pct,ln_input_index,ln_output_index
0,expertise,Claude.ai,1.000,0.461,0.494,-0.026,NaN,NaN
1,autonomy,Claude.ai,0.461,1.000,0.646,-0.142,NaN,NaN
2,human_only_time,Claude.ai,0.494,0.646,1.000,-0.500,NaN,NaN
3,task_success_pct,Claude.ai,-0.026,-0.142,-0.500,1.000,NaN,NaN
4,expertise,1P API,1.000,0.299,NaN,0.033,0.197,0.115
5,autonomy,1P API,0.299,1.000,NaN,-0.489,0.542,0.377
6,ln_input_index,1P API,0.197,0.542,NaN,-0.585,1.000,0.410
7,ln_output_index,1P API,0.115,0.377,NaN,-0.133,0.410,1.000
8,task_success_pct,1P API,0.033,-0.489,NaN,1.000,-0.585,-0.133


## Table D4 – Share of usage carried by the ten most-used tasks, by platform and release

The 1P API series is the 93% → 85% figure cited in Section 4.1.3 (Release 3 → Release 5).

In [6]:
rows = []
for r in rc.ANALYSIS_RELEASES:
    for p in rc.RELEASE_PLATFORMS[r]:
        share = rc.get_r2r_task_pct(panel, r, p)
        rows.append({"release": r, "platform": rc.PLATFORM_LABELS[p], "n_tasks_with_usage": int((share > 0).sum()),
                     "top10_share_pct": 100 * share.sort_values(ascending=False).head(10).sum()})
tableD4 = pd.DataFrame(rows).pivot(index="release", columns="platform", values="top10_share_pct").round(1)
rc.save_csv(tableD4.reset_index(), "appD", "tableD4_top10_task_share.csv")
tableD4

[write] outputs/4_appendix_D/tableD4_top10_task_share.csv


platform,1P API,Claude.ai
release,,
1,NaN,83.8
2,NaN,77.2
3,93.2,84.4
4,88.1,84.3
5,85.3,82.6


## Figure D3 – Distribution of human-only time by platform, Release 5

Supports the claim in Section 4.2.4 that API tasks are shorter and more homogeneous in duration (lower mean and lower coefficient of variation).

In [7]:
release = 5
data = {p: rc.get_task_metric(panel, release, p, "human_only_time").dropna() for p in rc.PLATFORMS}
summary = pd.DataFrame({rc.PLATFORM_LABELS[p]: {"n": len(s), "mean_h": s.mean(), "median_h": s.median(), "sd_h": s.std(),
                                                 "cv": s.std() / s.mean(), "iqr_h": s.quantile(.75) - s.quantile(.25)}
                        for p, s in data.items()}).T.round(3)
rc.save_csv(summary.reset_index().rename(columns={"index": "platform"}), "appD", "figD3_human_only_time_summary_R5.csv")
print(summary)

fig, ax = plt.subplots(figsize=(9, 5))
rng = np.random.default_rng(7)
for i, p in enumerate(rc.PLATFORMS):
    s = data[p]
    ax.boxplot(s, positions=[i], orientation="horizontal", widths=0.5, showfliers=False, patch_artist=True,
               boxprops=dict(facecolor="white", edgecolor=rc.PLATFORM_COLORS[p], linewidth=1.6),
               medianprops=dict(color=rc.PLATFORM_COLORS[p], linewidth=2),
               whiskerprops=dict(color=rc.PLATFORM_COLORS[p]), capprops=dict(color=rc.PLATFORM_COLORS[p]))
    ax.scatter(s, i + rng.uniform(-0.15, 0.15, len(s)), s=28, color=rc.PLATFORM_COLORS[p], alpha=0.75,
               marker=rc.PLATFORM_MARKERS[p], edgecolor="white", linewidth=0.5, zorder=3)
    ax.annotate(f"mean {s.mean():.2f} h | CV {s.std() / s.mean():.2f} | n = {len(s)}", (1.0, i),
                xycoords=("axes fraction", "data"), xytext=(6, 0), textcoords="offset points", va="center",
                ha="left", fontsize=9, color=rc.PLATFORM_COLORS[p], fontweight="semibold", annotation_clip=False)
ax.set_xscale("log")
ax.set_yticks(range(len(rc.PLATFORMS)))
ax.set_yticklabels([rc.PLATFORM_LABELS[p] for p in rc.PLATFORMS])
ax.set_xlabel("Human-only time (hours, log scale)")
rc.clean_axes(ax, grid="x")
ax.set_title(f"Human-only time by platform -- Release {release}", loc="left", fontsize=13)
fig.tight_layout()
rc.save_figure(fig, "appD", "figD3_human_only_time_by_platform_R5.png")

[write] outputs/4_appendix_D/figD3_human_only_time_summary_R5.csv
              n  mean_h  median_h   sd_h     cv  iqr_h
Claude.ai  38.0   3.571     3.331  1.612  0.451  2.391
1P API     28.0   1.570     1.118  1.285  0.819  1.486


findfont: Failed to find font weight semibold for Arial, now using 700.


[write] outputs/4_appendix_D/figD3_human_only_time_by_platform_R5.png
